<a href="https://colab.research.google.com/github/SaiDurga98/Machine-Learning/blob/main/Building_ANN_with_optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ANN architecture:
i/p layer -> 784 nodes (784 features)
hidden layer 1 -> 128 neurons -> reLU
hidden layer 2 -> 64 features -> reLU
output -> 10 neurons -> softwax since its a multi class classification problem


# Workflow:
1. Create dataloader objects for both taining and test data
2. Write training loop
3. Evaluate model using test data

In [38]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim
import matplotlib.pyplot as plt

In [39]:
# sets the starting point for Pytorch random number generator
# so the outcome is the same every time you run your code
torch.manual_seed(42)

In [40]:
# check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [41]:
# About datset:Each row is one image of a clothing item (28 *28 = 784 pixels grayscale)
# The 2D image is flattened into a single row so pixel one through pixel 784 are the columns
# Plus the label column = 785 total columns . The label is what you're predicting : 0-9 each number being a clothing type
# The pixel values are 0-255 where 0 is black and 255 is white. The most values in the data are 0 because edges of each image are just black and actual clothing shape sits in the middle.
# This 28 * 28 is theat will be feeded to our neural network i.e.. the input layer will need 784 neurons - one per pixel
#df = pd.read_csv('fmnist_small.csv') # just small dataset with few samples
df = pd.read_csv('fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [42]:
df.shape

(60000, 785)

In [43]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [45]:
# Scaling the features
# Dividing by 255(their max pixel value) squashes every pixel from the 0-255 range down to 0-1.
# This is called normalization. Neural n/w learns better with small numbers.
#Training works by multiplying inputs with weights and nudging those weights via gradients. If inputs are big (like 255), the multiplications produce large values, gradients can swing wildly, and training becomes unstable or slow — like trying to parallel park a car whose steering wheel is way too sensitive. With inputs between 0 and 1, the updates are smooth and controlled.

X_train = X_train/255.0
X_test = X_test/255.0

In [46]:
# Create CustomDataset class
class CustomDataset(Dataset):

  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype = torch.float32)
    self.labels = torch.tensor(labels, dtype = torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

In [47]:
# Create train_dataset object
train_dataset = CustomDataset(X_train, y_train)

In [14]:
len(train_dataset)

48000

In [ ]:
train_dataset[0]

In [48]:
test_dataset = CustomDataset(X_test, y_test)

In [18]:
len(test_dataset)

12000

In [19]:
# create train and test dataLoader objects
# pin_memory=True is a speed optimization for when you're training on a GPU.
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)


In [49]:
# Define Neural network class
class MyNeuralNetwork(nn.Module):

  def __init__(self, input_dim, output_dim, num_hidden_layers, num_neurons_per_layer, dropout_rate):
    super().__init__()

    layers = []
    for i in range(num_hidden_layers):
      layers.append(nn.Linear(input_dim, num_neurons_per_layer))
      layers.append(nn.BatchNorm1d(num_neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim = num_neurons_per_layer

    layers.append(nn.Linear(num_neurons_per_layer, output_dim))

    self.model = nn.Sequential(*layers)


  def forward(self, x):
    return self.model(x)

In [50]:
# Objective function
def objective(trial):

  # next hyperparameter values from search space
  num_hidden_layers = trial.suggest_int('num_hidden_layers', 1, 5)
  num_neurons_per_layer = trial.suggest_int('num_neurons_per_layer', 8, 128, step=8)
  epochs = trial.suggest_int('epochs', 10, 50, step=10)
  learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
  dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.1)
  batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
  optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
  weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

  train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
  test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)


  # model initialization
  input_dim = 784
  output_dim = 10
  model = MyNeuralNetwork(input_dim, output_dim, num_hidden_layers, num_neurons_per_layer, dropout_rate)
  model.to(device)

  # # Parameters init
  # epochs = 5
  # learning_rate = 0.1

  # Loss Function
  criterion = nn.CrossEntropyLoss()

  # Optimizer # weight_decay is regularization coefficients
  optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  if optimizer_name == 'Adam':
    optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  elif optimizer_name == 'SGD':
    optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  else:
    optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  # Training loop
  for epoch in range(epochs):

    total_epoch_loss = 0

    for batch_features, batch_labels in train_dataloader:
      # Move data to GPU
      batch_features = batch_features.to(device)
      batch_labels = batch_labels.to(device)

      # forward pass
      outputs = model(batch_features)

      # calculate loss
      loss = criterion(outputs, batch_labels)

      # back pass
      optimizer.zero_grad()
      loss.backward()

      # update grads
      optimizer.step()

      total_epoch_loss = total_epoch_loss + loss.item()

    avg_loss = total_epoch_loss/len(train_dataloader)
    print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

    # Evaluation
    model.eval()

  total = 0
  correct = 0
  with torch.no_grad():
    for batch_features, batch_labels in test_dataloader:
      # Move data to GPU
      batch_features = batch_features.to(device)
      batch_labels = batch_labels.to(device)

      # Apply forward pass for prediction values
      outputs = model(batch_features)
      _, predicted = torch.max(outputs, 1)
      total += batch_labels.shape[0] # (32+32+32...)
      correct += (predicted == batch_labels).sum().item()

  accuracy = 100 * correct / total
  print(f'Accuracy: {accuracy}')

  return accuracy

In [35]:
!pip install optuna

In [51]:
import optuna

study = optuna.create_study(direction='maximize')


[I 2026-07-29 00:56:41,942] A new study created in memory with name: no-name-7c8ed0ac-29b2-4699-a028-dcf1074aece8


In [ ]:
study.optimize(objective, n_trials=10)

Epoch: 1 , Loss: 1.7253580018679302
Epoch: 2 , Loss: 0.9729980745315552
Epoch: 3 , Loss: 0.7628207294940949
Epoch: 4 , Loss: 0.687384637872378
Epoch: 5 , Loss: 0.6425579224030177
Epoch: 6 , Loss: 0.6114878039757411
Epoch: 7 , Loss: 0.5882257217963537
Epoch: 8 , Loss: 0.5696266025702159
Epoch: 9 , Loss: 0.5549706426858902


[W 2026-07-29 00:56:58,115] Trial 0 failed with parameters: {'num_hidden_layers': 1, 'num_neurons_per_layer': 56, 'epochs': 10, 'learning_rate': 0.0004726227827171322, 'dropout_rate': 0.4, 'batch_size': 64, 'optimizer': 'SGD', 'weight_decay': 7.961210033430027e-05} because of the following error: The value None could not be cast to float..
[W 2026-07-29 00:56:58,116] Trial 0 failed with value None.


Epoch: 10 , Loss: 0.5427550905545553
Accuracy: 81.10833333333333
Epoch: 1 , Loss: 2.0006444082260133
Epoch: 2 , Loss: 0.7154577374855677
Epoch: 3 , Loss: 0.5031351564129194
Epoch: 4 , Loss: 0.45011686990658445
Epoch: 5 , Loss: 0.4220871509114901
Epoch: 6 , Loss: 0.39990228913227716
Epoch: 7 , Loss: 0.38608284057180087
Epoch: 8 , Loss: 0.3713837116161982
Epoch: 9 , Loss: 0.360241092701753
Epoch: 10 , Loss: 0.35177363828818003
Epoch: 11 , Loss: 0.3424875933031241
Epoch: 12 , Loss: 0.3345784785846869
Epoch: 13 , Loss: 0.32856547556320825
Epoch: 14 , Loss: 0.3228240454395612
Epoch: 15 , Loss: 0.31654824856917063
Epoch: 16 , Loss: 0.3111410807073116
Epoch: 17 , Loss: 0.3061148300766945
Epoch: 18 , Loss: 0.3009212309618791
Epoch: 19 , Loss: 0.29625397876898446
Epoch: 20 , Loss: 0.2924430939455827
Epoch: 21 , Loss: 0.28680984236796697
Epoch: 22 , Loss: 0.2836537695924441
Epoch: 23 , Loss: 0.2799264408747355
Epoch: 24 , Loss: 0.2757975246012211
Epoch: 25 , Loss: 0.2733486541012923
Epoch: 26 , 

[W 2026-07-29 00:58:41,308] Trial 1 failed with parameters: {'num_hidden_layers': 5, 'num_neurons_per_layer': 64, 'epochs': 40, 'learning_rate': 0.0016162775057980841, 'dropout_rate': 0.2, 'batch_size': 64, 'optimizer': 'Adam', 'weight_decay': 0.0004585312417488787} because of the following error: The value None could not be cast to float..
[W 2026-07-29 00:58:41,309] Trial 1 failed with value None.


Accuracy: 88.29166666666667
Epoch: 1 , Loss: 0.7578083880345027
Epoch: 2 , Loss: 0.4471382704426845
Epoch: 3 , Loss: 0.3964225283761819
Epoch: 4 , Loss: 0.36931803525984286
Epoch: 5 , Loss: 0.3527811064918836
Epoch: 6 , Loss: 0.3387020986303687
Epoch: 7 , Loss: 0.3267125329027573
Epoch: 8 , Loss: 0.31670221320539715
Epoch: 9 , Loss: 0.3075782431438565
Epoch: 10 , Loss: 0.29930630917847156
Epoch: 11 , Loss: 0.2938409175326427
Epoch: 12 , Loss: 0.2862799808407823
Epoch: 13 , Loss: 0.279466298888127
Epoch: 14 , Loss: 0.27582236018776896
Epoch: 15 , Loss: 0.2688436307609081
Epoch: 16 , Loss: 0.2645315315996607
Epoch: 17 , Loss: 0.2598026443682611
Epoch: 18 , Loss: 0.2554421114201347
Epoch: 19 , Loss: 0.2498380085254709
Epoch: 20 , Loss: 0.2464090755780538
Epoch: 21 , Loss: 0.24277274861807624
Epoch: 22 , Loss: 0.2393845571950078
Epoch: 23 , Loss: 0.23480673388888437
Epoch: 24 , Loss: 0.23173304616287352
Epoch: 25 , Loss: 0.2266390659424166
Epoch: 26 , Loss: 0.22557315941775838
Epoch: 27 , 

...

Accuracy: 87.61666666666666
Epoch: 1 , Loss: 0.973945838590463
Epoch: 2 , Loss: 0.5029832887699207
Epoch: 3 , Loss: 0.4474004305253426
Epoch: 4 , Loss: 0.41895955576002597
Epoch: 5 , Loss: 0.4001886250500878
Epoch: 6 , Loss: 0.3858595447217425
Epoch: 7 , Loss: 0.37471343759509423
Epoch: 8 , Loss: 0.3643340533121179
Epoch: 9 , Loss: 0.3563848502896726
Epoch: 10 , Loss: 0.3491023935141663
Epoch: 11 , Loss: 0.3421611257592837
Epoch: 12 , Loss: 0.33480766279560825
Epoch: 13 , Loss: 0.3288375125353535
Epoch: 14 , Loss: 0.3236795734409243
Epoch: 15 , Loss: 0.31769129145145414
Epoch: 16 , Loss: 0.31282267133705316
Epoch: 17 , Loss: 0.3085184058342129
Epoch: 18 , Loss: 0.3040488946909706
Epoch: 19 , Loss: 0.29960685700240236
Epoch: 20 , Loss: 0.29511495743008953


[W 2026-07-29 01:06:31,907] Trial 4 failed with parameters: {'num_hidden_layers': 1, 'num_neurons_per_layer': 96, 'epochs': 20, 'learning_rate': 0.0015184200598260752, 'dropout_rate': 0.4, 'batch_size': 16, 'optimizer': 'RMSprop', 'weight_decay': 1.5080954156506265e-05} because of the following error: The value None could not be cast to float..
[W 2026-07-29 01:06:31,908] Trial 4 failed with value None.


Accuracy: 87.85833333333333
Epoch: 1 , Loss: 2.391502847035726
Epoch: 2 , Loss: 2.2711579186121624
Epoch: 3 , Loss: 2.2183039989471434
Epoch: 4 , Loss: 2.173414145787557
Epoch: 5 , Loss: 2.12917014837265
Epoch: 6 , Loss: 2.0823478816350303
Epoch: 7 , Loss: 2.0324581576983136
Epoch: 8 , Loss: 1.979127935409546
Epoch: 9 , Loss: 1.923751779874166
Epoch: 10 , Loss: 1.868428075949351


[W 2026-07-29 01:06:52,397] Trial 5 failed with parameters: {'num_hidden_layers': 3, 'num_neurons_per_layer': 48, 'epochs': 10, 'learning_rate': 1.049993706761079e-05, 'dropout_rate': 0.30000000000000004, 'batch_size': 64, 'optimizer': 'SGD', 'weight_decay': 7.130719579950798e-05} because of the following error: The value None could not be cast to float..
[W 2026-07-29 01:06:52,398] Trial 5 failed with value None.


Accuracy: 47.28333333333333
Epoch: 1 , Loss: 1.1512159765561423
Epoch: 2 , Loss: 0.5061448132197062
Epoch: 3 , Loss: 0.4340531448523203
Epoch: 4 , Loss: 0.4015056505997976
Epoch: 5 , Loss: 0.3778622723420461
Epoch: 6 , Loss: 0.36185944414138793
Epoch: 7 , Loss: 0.34887545589605967
Epoch: 8 , Loss: 0.3361163118282954
Epoch: 9 , Loss: 0.3288051890532176


In [ ]:
study.best_value
study.best_params

# The below is the code for ANN before applying optuna

In [44]:
# Define Neural network class
class MyNeuralNetwork(nn.Module):

  def __init__(self, features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(features, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(128, 64), #output of first layer goes into next
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(64, 10)
        # pytorch handles softmax implicitly at output layer so not need to apply softmax
    )

  def forward(self, x):
    return self.model(x)



In [45]:
# Set learning rate and epochs
epochs = 100
learning_rate = 0.1

In [46]:
# Instantiate model
model = MyNeuralNetwork(X_train.shape[1])
# Move model to GPU
model.to(device)
# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer # weight_decay is regularization coefficients
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)




In [47]:
# Training Loop

for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_dataloader:
    # Move data to GPU
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # calculate loss
    loss = criterion(outputs, batch_labels)

    # back pass
    optimizer.zero_grad()
    loss.backward()

    # update grads
    optimizer.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_loss = total_epoch_loss/len(train_dataloader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 0.6249084657629331
Epoch: 2 , Loss: 0.49199690653880435
Epoch: 3 , Loss: 0.45562089485426743
Epoch: 4 , Loss: 0.43380642544229825
Epoch: 5 , Loss: 0.41715061584611735
Epoch: 6 , Loss: 0.40564093277355034
Epoch: 7 , Loss: 0.3941608931571245
Epoch: 8 , Loss: 0.38580174928406874
Epoch: 9 , Loss: 0.3743983890265226
Epoch: 10 , Loss: 0.3725726637095213
Epoch: 11 , Loss: 0.36783315147956214
Epoch: 12 , Loss: 0.3572052289446195
Epoch: 13 , Loss: 0.35052060889204345
Epoch: 14 , Loss: 0.3449219484726588
Epoch: 15 , Loss: 0.34472562207778296
Epoch: 16 , Loss: 0.33732124184072015
Epoch: 17 , Loss: 0.3344038988550504
Epoch: 18 , Loss: 0.3302020480086406
Epoch: 19 , Loss: 0.33063985937833784
Epoch: 20 , Loss: 0.3262277270356814
Epoch: 21 , Loss: 0.3208496819138527
Epoch: 22 , Loss: 0.3183093272894621
Epoch: 23 , Loss: 0.3225850373158852
Epoch: 24 , Loss: 0.31459670132398604
Epoch: 25 , Loss: 0.31343053522954384
Epoch: 26 , Loss: 0.31424527982374034
Epoch: 27 , Loss: 0.3107087447295

In [49]:
# Set model to eval mode
model.eval()

MyNeuralNetwork(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [51]:
# Evaluation code
total = 0
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in test_dataloader:
     # Move data to GPU
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    # Apply forward pass for prediction values
    outputs = model(batch_features)
    _, predicted = torch.max(outputs, 1)
    total += batch_labels.shape[0] # (32+32+32...)
    correct += (predicted == batch_labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy: {accuracy}')



Accuracy: 89.10833333333333


# **How the accuracy above is calculated:**

 You send in 1 batch of 32 test images → the model returns a tensor of shape (32, 10) — one row per image, 10 scores per row (one for each clothing class). Each row's highest-score position is the predicted label.

So the key idea: the model doesn't answer "7" directly. It answers with 10 raw scores (called logits), and you pick the winner — usually with torch.max(outputs, 1) or argmax, which returns the index of the largest score in each row. That index is the predicted class.

That's exactly what your evaluation loop will do next: compare those 32 argmax picks against the 32 true labels, count matches, and that's your accuracy.

# Optimizing neural network:
**Overfitting** occurs when model's accuracy is good on training data but not that accurate on test data.

Optimization techniques:

1.Regularization
2.Dropout
3.Batch Normalization

# Batch Normalization
1. Applied after linear layer and before activation function

Your input data is nicely scaled between 0 and 1 — but only at the front door. After the data passes through the first layer (multiplied by weights, summed, ReLU'd), the numbers coming out can be all over the place — some huge, some tiny. And it gets worse: as training updates the weights of layer 1, the distribution of values flowing into layer 2 keeps shifting every single step. Layer 2 is trying to learn from an input that keeps changing its scale and center — like trying to hit a moving target. This shifting is often called internal covariate shift.

Batch norm fixes it by re-standardizing the activations at each layer, for each batch: take the batch's values at that layer, subtract the mean, divide by the standard deviation — so they're re-centered around 0 with a consistent spread — then let the network scale/shift them via two small learnable parameters (gamma and beta) if it prefers something other than exactly 0-mean-1-std.